# Terceiro Projeto: Construindo a Árvore de Sintaxe Abstrata

Uma árvore de sintaxe abstrata (do inglês, *Abstract Syntax Tree* - AST) é uma estrutura de dados que representa melhor a estrutura do código do programa do que a árvore de sintaxe. Uma AST pode ser editada e aprimorada com informações como propriedades e anotações para cada elemento que ela contém. Seu objetivo neste terceiro projeto é transformar a árvore de sintaxe em uma AST.

## Árvore de Sintaxe Abstrata

Já mostramos como analisar uma gramática e gerar uma estrutura em forma de árvore a partir dela. Tuplas, no entanto, são um pouco limitadas. Neste terceiro projeto, você irá atualizar para classes.

Cada nó na árvore será representado por uma classe. Sua tarefa é usar o analisador sintático para conectar essas classes/nós e construir efetivamente a AST.

Esta seção define classes para diferentes tipos de nós de uma AST. Durante a análise sintática, você criará esses nós e os conectará. Em geral, você terá um nó AST diferente para cada tipo de regra gramatical.

Apenas como exemplo, para um operador binário, você pode armazenar o operador, a expressão esquerda e a expressão direita assim:

In [ ]:
class BinaryOp:
    def __init__(self, op, left, right):
        self.op = op
        self.left = left
        self.right = right

    def __repr__(self):
        return f'BinaryOp({self.op}, {self.left}, {self.right})'

Para um literal, você pode armazenar o tipo e o valor, assim:

In [ ]:
class Literal:
    def __init__(self, dtype, value):
        self.dtype = dtype
        self.value = value

    def __repr__(self):
        return f'Literal({self.dtype}, {self.value!r})'

**Sugestão:** Você deve começar de forma simples e trabalhar incrementalmente até construir a gramática completa.

## Mostrando a AST

Sempre que uma regra de substituição é totalmente reconhecida pelo analisador sintático, uma ação semântica é executada. Com esse conhecimento, podemos retornar nós AST que representam a derivação/redução em questão em vez de tuplas simples.

Considere como exemplo a seguinte expressão:

```
2 + 3 * 4
```

A AST gerada pelo analisador sintático para essa expressão pode ser codificada manualmente instanciando diretamente suas classes/nós:

In [ ]:
model = BinaryOp('+', Literal('i32', 2),
                      BinaryOp('*', Literal('i32', 3),
                                    Literal('i32', 4)))

A impressão da AST para o exemplo acima é:

In [ ]:
print(model)

Você consegue transformar a AST novamente em código-fonte? Para facilitar a visualização, podemos escrever uma função de depuração para converter uma AST em código-fonte:

In [ ]:
def to_source(node, indent=''):
    if isinstance(node, Literal):
        return f'{node.value!r}'
    elif isinstance(node, BinaryOp):
        return f'{to_source(node.left)} {node.op} {to_source(node.right)}'
    else:
        raise RuntimeError(f"Can't convert {node} to source")

In [ ]:
print(to_source(model))

## Esboço da Construção da AST

Complete o código abaixo, escrevendo classes de nós AST para cada construção da linguagem uZig.

In [ ]:
class BinaryOp:
    def __init__(self, op, left, right):
        self.op = op
        self.left = left
        self.right = right

    def __repr__(self):
        return f'BinaryOp({self.op}, {self.left}, {self.right})'

class Literal:
    def __init__(self, dtype, value):
        self.dtype = dtype
        self.value = value

    def __repr__(self):
        return f'Literal({self.dtype}, {self.value!r})'

# ------ Debugging function to convert a model into source code (for easier viewing)

def to_source(node, indent=''):
    if isinstance(node, Literal):
        return f'{node.value!r}'
    elif isinstance(node, BinaryOp):
        return f'{to_source(node.left)} {node.op} {to_source(node.right)}'
    else:
        raise RuntimeError(f"Can't convert {node} to source")

In [ ]:
!pip install sly

Copie o código do analisador léxico que você escreveu no [primeiro projeto](https://colab.research.google.com/drive/1JYNeH16cMf46jRF8JGYBRGeCQng8LZhO?usp=sharing) e cole na célula abaixo.

In [ ]:
# High level function that takes input source text and turns it into tokens.
# This is a natural place to use some kind of generator function.
from sly import Lexer

class zigLexer(Lexer):
    tokens = {
        'KEYWORD_and',
        'KEYWORD_break',
        'KEYWORD_const',
        'KEYWORD_continue',
        'KEYWORD_else',
        'KEYWORD_false',
        'KEYWORD_if',
        'KEYWORD_or',
        'KEYWORD_true',
        'KEYWORD_var',
        'KEYWORD_while',
        'IDENTIFIER',
        'BUILTINIDENTIFIER',
        'INTEGER',
        'FLOAT',
        'CHAR_LITERAL',
        'STRINGLITERAL',
        'EQUALEQUAL',
        'EXCLAMATIONMARKEQUAL',
        'LARROWEQUAL',
        'RARROWEQUAL',
        'LARROW',
        'RARROW',
        'PLUS',
        'MINUS',
        'ASTERISK',
        'SLASH',
        'PERCENT',
        'EQUAL',
        'EXCLAMATIONMARK',
        'COLON',
        'LPAREN',
        'RPAREN',
        'LBRACE',
        'RBRACE',
        'COMMA',
        'SEMI'
    }

    EQUALEQUAL = r'=='
    EXCLAMATIONMARKEQUAL = r'!='
    LARROWEQUAL = r'<='
    RARROWEQUAL = r'>='
    PLUS = r'\+'
    MINUS = r'-'
    ASTERISK = r'\*'
    PERCENT = r'%'
    EQUAL = r'='
    LARROW = r'<'
    RARROW = r'>'
    EXCLAMATIONMARK = r'!'
    COLON = r':'
    LPAREN = r'\('
    RPAREN = r'\)'
    LBRACE = r'\{'
    RBRACE = r'\}'
    COMMA = r','
    SEMI = r';'
    FLOAT = r'\d*\.\d+|\d+\.'
    INTEGER = r'\d+'
    BUILTINIDENTIFIER = r'@[a-zA-Z_][0-9a-zA-Z_]*'
    ignore = ' \t'

    @_(r'[a-zA-Z_][0-9a-zA-Z_]*')
    def IDENTIFIER(self, t):
        keywords = {
            'and', 'break', 'const', 'continue', 'else',
            'false', 'if', 'or', 'true', 'var', 'while'
        }
        if t.value in keywords:
            t.type = 'KEYWORD_' + t.value
        return t

    @_(r'\n+')
    def ignore_newline(self, t):
        self.lineno += t.value.count('\n')

    @_(r'//.*')
    def ignore_comment(self, t):
        pass

    @_(r'/')
    def SLASH(self, t):
        return t

    @_(r'"(?:\\.|[^"\\])*"')
    def STRINGLITERAL(self, t):
        self.lineno += t.value.count('\n')
        return t

    @_(r'"(?:\\.|[^"\\])*')
    def STRING_UNTERMINATED(self, t):
        print(f'{self.lineno}: Unterminated string literal')
        self.lineno += t.value.count('\n')

    @_(r"'(?:\\.|[^'\\\n])*'")
    def CHAR_LITERAL(self, t):
        return t

    @_(r"'(?:\\.|[^'\\\n])*")
    def CHAR_UNTERMINATED_ERROR(self, t):
        print(f'{self.lineno}: Unterminated character constant')

    def error(self, t):
        print(f'{self.lineno}: Illegal character \'{t.value[0]}\'')
        self.index += 1

def tokenize(text):
    lexer = zigLexer()
    yield from lexer.tokenize(text)


Copie o código do analisador sintático que você escreveu no [segundo projeto](https://colab.research.google.com/drive/1l7j56w4OQUH_pIomkbIiOGzQwmofegd6?usp=sharing) e cole na célula abaixo. Em seguida, modifique as regras para retornarem nós AST em vez de tuplas.

In [ ]:
# High level function that takes input tokens and turns it into a syntax tree.
# This is a natural place to use some kind of generator function.

import sys
from sly import Parser
#from uzig_lexer import zigLexer


def _node(tag, *children):
    return (tag,) + children


class zigParser(Parser):

    tokens = zigLexer.tokens
    precedence = ( ('nonassoc', 'LOWER_THAN_ELSE'), ('nonassoc', 'KEYWORD_else'), ('left', 'KEYWORD_or'), ('left', 'KEYWORD_and'), ('left', 'EQUALEQUAL', 'EXCLAMATIONMARKEQUAL'), ('left', 'LARROW', 'LARROWEQUAL', 'RARROW', 'RARROWEQUAL'), ('left', 'PLUS', 'MINUS'), ('left', 'ASTERISK', 'SLASH', 'PERCENT'), ('right', 'UMINUS', 'UPLUS', 'EXCLAMATIONMARK'),
    )

    @_('statement_list')
    def program(self, p):
        return _node('program', p.statement_list)

    @_('statement_list statement')
    def statement_list(self, p):
        return p.statement_list + [p.statement]

    @_('statement')
    def statement_list(self, p):
        return [p.statement]

    @_('assignment_statement',
       'variable_definition',
       'const_definition',
       'if_statement',
       'while_statement',
       'break_statement',
       'continue_statement',
       'expression_statement')
    def statement(self, p):
        return p[0]

    @_('location EQUAL expression SEMI')
    def assignment_statement(self, p):
        return _node('assignment', p.location, p.expression)

    @_('KEYWORD_var IDENTIFIER COLON type EQUAL expression SEMI')
    def variable_definition(self, p):
        return _node(f'variable: {p.IDENTIFIER}', p.type, p.expression)

    @_('KEYWORD_var IDENTIFIER COLON type SEMI')
    def variable_definition(self, p):
        return _node(f'variable: {p.IDENTIFIER}', p.type, 'None')

    @_('KEYWORD_var IDENTIFIER EQUAL expression SEMI')
    def variable_definition(self, p):
        return _node(f'variable: {p.IDENTIFIER}', 'None', p.expression)

    @_('KEYWORD_const IDENTIFIER COLON type EQUAL expression SEMI')
    def const_definition(self, p):
        return _node(f'const: {p.IDENTIFIER}', p.type, p.expression)

    @_('KEYWORD_const IDENTIFIER EQUAL expression SEMI')
    def const_definition(self, p):
        return _node(f'const: {p.IDENTIFIER}', 'None', p.expression)

    @_('KEYWORD_if LPAREN expression RPAREN block %prec LOWER_THAN_ELSE')
    def if_statement(self, p):
        return _node('if', p.expression, p.block, 'None')

    @_('KEYWORD_if LPAREN expression RPAREN block KEYWORD_else block')
    def if_statement(self, p):
        return _node('if', p.expression, p.block0, p.block1)

    @_('KEYWORD_if LPAREN expression RPAREN block KEYWORD_else if_statement')
    def if_statement(self, p):
        return _node('if', p.expression, p.block, p.if_statement)

    @_('KEYWORD_while LPAREN expression RPAREN block')
    def while_statement(self, p):
        return _node('while', p.expression, p.block)

    @_('KEYWORD_break SEMI')
    def break_statement(self, p):
        return 'break'

    @_('KEYWORD_continue SEMI')
    def continue_statement(self, p):
        return 'continue'

    @_('expression SEMI')
    def expression_statement(self, p):
        return _node('expression', p.expression)

    @_('SEMI')
    def expression_statement(self, p):
        return _node('expression', 'None')

    @_('LBRACE statement_list RBRACE')
    def block(self, p):
        return _node('block', p.statement_list)

    @_('LBRACE RBRACE')
    def block(self, p):
        return _node('block', 'None')

    @_('expression PLUS expression',
       'expression MINUS expression',
       'expression ASTERISK expression',
       'expression SLASH expression',
       'expression PERCENT expression',
       'expression LARROWEQUAL expression',
       'expression LARROW expression',
       'expression RARROWEQUAL expression',
       'expression RARROW expression',
       'expression EQUALEQUAL expression',
       'expression EXCLAMATIONMARKEQUAL expression',
       'expression KEYWORD_and expression',
       'expression KEYWORD_or expression')
    def expression(self, p):
        return _node(f'binary_op: {p[1]}', p.expression0, p.expression1)

    @_('PLUS expression %prec UPLUS',
       'MINUS expression %prec UMINUS',
       'EXCLAMATIONMARK expression')
    def expression(self, p):
        return _node(f'unary_op: {p[0]}', p.expression)

    @_('literal')
    def expression(self, p):
        return p.literal

    @_('location')
    def expression(self, p):
        return p.location

    @_('BUILTINIDENTIFIER LPAREN expression_list RPAREN')
    def expression(self, p):
        return _node(f'builtin: {p.BUILTINIDENTIFIER}', p.expression_list)

    @_('BUILTINIDENTIFIER LPAREN RPAREN')
    def expression(self, p):
        return _node(f'builtin: {p.BUILTINIDENTIFIER}', [])

    @_('LPAREN expression RPAREN')
    def expression(self, p):
        return p.expression

    @_('INTEGER')
    def literal(self, p):
        return f"literal: i32, {p.INTEGER}"

    @_('FLOAT')
    def literal(self, p):
        return f"literal: f64, {p.FLOAT}"

    @_('KEYWORD_true', 'KEYWORD_false')
    def literal(self, p):
        return f"literal: bool, {p[0]}"

    @_('STRINGLITERAL')
    def literal(self, p):
        return f"literal: []const u8, {p.STRINGLITERAL}"

    @_('CHAR_LITERAL')
    def literal(self, p):
        return f"literal: u8, {p.CHAR_LITERAL}"

    @_('expression')
    def expression_list(self, p):
        return [p.expression]

    @_('expression_list COMMA expression')
    def expression_list(self, p):
        return p.expression_list + [p.expression]

    @_('IDENTIFIER')
    def location(self, p):
        return f"location: {p.IDENTIFIER}"

    @_('IDENTIFIER')
    def type(self, p):
        return f"type: {p.IDENTIFIER}"

    def error(self, token):
        if token:
            print(f"Syntax error at line {token.lineno}, token={token.type}")
        else:
            print("Parse error in input. EOF")


def parse_tokens(token_stream):
    return zigParser().parse(token_stream)

def _collect_lines(node, lines, indent, last):
    branch = '└── ' if last else '├── '
    extension = '    ' if last else '│   '
    if isinstance(node, list):
        if not node:
            return
        node = tuple(node)
    if not isinstance(node, tuple):
        lines.append(indent + branch + str(node))
        return
    label, *children = node
    lines.append(indent + branch + str(label))
    for i, child in enumerate(children):
        _collect_lines(child, lines, indent + extension, i == len(children) - 1)

def build_tree(root):
    lines = []
    _collect_lines(root, lines, '', True)
    return '\n'.join(lines)


In [ ]:
# Top-level function that runs everything
def parse_source(text):
    tokens = tokenize(text)
    model = parse_tokens(tokens)     # You need to implement this part
    return model

In [ ]:
# Main program to test on input files
def main(filename):
    with open(filename) as file:
        text = file.read()

    model = parse_source(text)
    if model:
        print(to_source(model))

## Teste
Para o desenvolvimento inicial, tente executar o analisador sintático em um arquivo de entrada de exemplo, como:

```
// print values of factorials
var n : i32 = 1;
var value : i32 = 1;

while (n < 10)
{
	value = value * n;
	@print(value);
	n = n + 1;
}
```

In [ ]:
%%file test.uzig
// print values of factorials
var n : i32 = 1;
var value : i32 = 1;

while (n < 10)
{
	value = value * n;
	@print(value);
	n = n + 1;
}

E o resultado será semelhante ao texto mostrado abaixo.

```
var n : i32 = 1;
var value : i32 = 1;
while ( n < 10 )
{
    value = value * n;
    @print( value );
    n = n + 1;
}
```

In [ ]:
main(["test.uzig"])

## Envie seu trabalho

Depois de concluir esta tarefa, salve o código das [classes de nós AST](#scrollTo=okQJM6uCeBnv) em um arquivo chamado `uzig_ast.py`, salve o código do seu [analisador léxico](#scrollTo=NR5AOrI7d91P) em um arquivo chamado `uzig_lexer.py`, copie o código do seu [analisador sintático](#scrollTo=wjdOC2uKiRld) e o submeta no [AVA](https://ava.ufscar.br/mod/quiz/view.php?id=1035124).